### ETL Bronze — Financial Transactions Dataset

This notebook ingests the 5 raw data sources (transactions, cards, users, mcc_codes, fraud_labels) 
and saves them as Delta tables in the bronze schema.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS jarvis_databricks.bronze;
CREATE SCHEMA IF NOT EXISTS jarvis_databricks.silver;
CREATE SCHEMA IF NOT EXISTS jarvis_databricks.gold;

#### JDBC Ingestion — Azure SQL Database

In [0]:
username = dbutils.secrets.get(scope="jdbc", key="username")
password = dbutils.secrets.get(scope="jdbc", key="password")

url = "jdbc:sqlserver://jarvis-sql-samia.database.windows.net:1433;databaseName=free-sql-db-1966735;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

transactions_df = (spark.read
    .format("jdbc")
    .option("url", url)
    .option("dbtable", "dbo.transactions_data")
    .option("user", username)
    .option("password", password)
    .load()
)

display(transactions_df.limit(10))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
9099414,2011-02-01T12:04:00Z,1276,2009,117.4600,Swipe Transaction,48919,Chicago,IL,60618.0,5311,null
9099415,2011-02-01T12:04:00Z,1370,4957,126.0400,Swipe Transaction,40078,Elk Grove,CA,95624.0,5300,null
9099416,2011-02-01T12:04:00Z,1645,5878,101.6700,Swipe Transaction,38602,Cleveland,OH,44121.0,5311,null
9099419,2011-02-01T12:05:00Z,89,2639,15.0800,Swipe Transaction,44678,Mesilla Park,NM,88047.0,5812,null
9099420,2011-02-01T12:05:00Z,634,2272,15.4000,Swipe Transaction,29988,Elk Grove,CA,95758.0,5411,null
9099421,2011-02-01T12:05:00Z,1500,4729,3.8400,Swipe Transaction,69972,Elk Grove,CA,95624.0,5814,null
9099422,2011-02-01T12:05:00Z,1654,2915,-78.0000,Swipe Transaction,59935,Stafford,TX,77477.0,5499,null
9099423,2011-02-01T12:06:00Z,335,5131,133.1700,Online Transaction,87530,ONLINE,null,null,4900,Bad Card Number
9099424,2011-02-01T12:06:00Z,400,3776,34.8300,Swipe Transaction,50783,Panama City,FL,32408.0,5411,null
9099425,2011-02-01T12:06:00Z,677,179,208.1200,Swipe Transaction,74067,Saint Paul,MN,55116.0,8021,null


#### ADLS Ingestion — External Location

In [0]:
cards_df = (spark.read
    .format("jdbc")
    .option("url", url)
    .option("dbtable", "dbo.cards_data")
    .option("user", username)
    .option("password", password)
    .load()
)

display(cards_df.limit(10))

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,04/2024,866,true,2,33900.0000,01/1991,2014,No
1,550,Mastercard,Credit,5278231764792292,06/2024,396,true,1,11600.0000,01/1994,2013,No
2,556,Mastercard,Debit,5889825928297675,09/2021,422,true,1,19948.0000,01/1995,2011,No
3,1937,Visa,Credit,4289888672554714,04/2020,736,true,2,16400.0000,01/1995,2015,No
4,1981,Mastercard,Debit,5433366978583845,03/2024,530,true,2,19439.0000,01/1997,2007,No
5,619,Visa,Debit,4657824650820465,04/2024,245,true,2,21883.0000,01/1997,2012,No
6,1046,Amex,Credit,394584924614148,02/1999,302,true,2,9400.0000,01/1998,2011,No
7,511,Mastercard,Debit,5585238056278288,03/2005,749,true,1,9664.0000,01/1998,2011,No
8,1107,Mastercard,Credit,5462760953855576,09/2021,665,false,2,10300.0000,01/1998,2006,No
9,1046,Amex,Credit,357982644067712,09/2020,72,true,1,13000.0000,01/1999,2005,No


In [0]:
users_df = (spark.read
    .format("csv")
    .option("header", "true")
    .load("abfss://bronze-data@jarvisfrauddata.dfs.core.windows.net/users_data.csv")
)
display(users_df.limit(10))

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,$20599,$41997,$0,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,$25258,$51500,$102286,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,$26790,$54623,$114711,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,$26273,$42509,$2895,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,$18730,$38190,$81262,810,1


In [0]:
import json

# Lire le fichier JSON brut comme texte depuis ADLS
mcc_raw = spark.read.option("multiline", "true").text("abfss://bronze-data@jarvisfrauddata.dfs.core.windows.net/mcc_codes.json")

# Récupérer le contenu complet en une seule chaîne
mcc_json_str = "".join([row.value for row in mcc_raw.collect()])
mcc_dict = json.loads(mcc_json_str)

# Transformer en DataFrame
mcc_rows = [(k, v) for k, v in mcc_dict.items()]
mcc_df = spark.createDataFrame(mcc_rows, ["mcc_code", "mcc_description"])

display(mcc_df.limit(10))

mcc_code,mcc_description
5812,Eating Places and Restaurants
5541,Service Stations
7996,"Amusement Parks, Carnivals, Circuses"
5411,"Grocery Stores, Supermarkets"
4784,Tolls and Bridge Fees
4900,"Utilities - Electric, Gas, Water, Sanitary"
5942,Book Stores
5814,Fast Food Restaurants
4829,Money Transfer
5311,Department Stores


In [0]:
fraud_raw = spark.read.option("multiline", "true").text("abfss://bronze-data@jarvisfrauddata.dfs.core.windows.net/train_fraud_labels.json")

fraud_json_str = "".join([row.value for row in fraud_raw.collect()])
fraud_dict = json.loads(fraud_json_str)

fraud_target = fraud_dict.get("target", fraud_dict)

fraud_rows = [(k, v == "Yes") for k, v in fraud_target.items()]
fraud_df = spark.createDataFrame(fraud_rows, ["transaction_id", "is_fraud"])

display(fraud_df.limit(10))

transaction_id,is_fraud
10649266,false
23410063,false
9316588,false
12478022,false
9558530,false
12532830,false
19526714,false
9906964,false
13224888,false
13749094,false


#### Save Bronze Tables

In [0]:
transactions_df.write.mode("overwrite").saveAsTable("jarvis_databricks.bronze.transactions_data_bronze")
cards_df.write.mode("overwrite").saveAsTable("jarvis_databricks.bronze.cards_data_bronze")
users_df.write.mode("overwrite").saveAsTable("jarvis_databricks.bronze.users_data_bronze")
mcc_df.write.mode("overwrite").saveAsTable("jarvis_databricks.bronze.mcc_codes_bronze")
fraud_df.write.mode("overwrite").saveAsTable("jarvis_databricks.bronze.train_fraud_labels_bronze")